# Data Audit

Проверка качества данных перед моделированием оттока: структура, пропуски, дубликаты, аномалии, согласованность полей.

Источник: `bank_churn_dataset.csv`, 80 000 клиентов, 26 признаков.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

df = pd.read_csv("../data/raw/bank_churn_dataset.csv")
print("Размер данных:", df.shape)
print("\nТипы данных:")
print(df.dtypes)

Размер данных: (80000, 26)

Типы данных:
id                          int64
full_name                     str
credit_sco                  int64
gender                        str
age                         int64
occupation                    str
balance                     int64
monthly_ir                  int64
address                       str
origin_province               str
tenure_ye                   int64
married                     int64
nums_card                   int64
nums_service                int64
active_member                bool
last_active_date              str
last_transaction_month      int64
created_date                  str
exit                         bool
customer_segment              str
engagement_score            int64
loyalty_level                 str
digital_behavior              str
risk_score                float64
risk_segment                  str
cluster_group               int64
dtype: object


In [2]:
df.head(10)

,id,full_name,credit_sco,gender,age,occupation,balance,monthly_ir,address,origin_province,tenure_ye,married,nums_card,nums_service,active_member,last_active_date,last_transaction_month,created_date,exit,customer_segment,engagement_score,loyalty_level,digital_behavior,risk_score,risk_segment,cluster_group
0,1,Đặng Văn Vũ,725,male,55,Chủ Doanh nghiệp nhỏ,177306004,121000000,Phường An Hội (BT),TP. Hồ Chí Minh,0,2,4,8,True,03/04/2025,91038993,27/02/2025,False,Priority,90,Bronze,mobile,0.0359,Low,4
1,2,Bùi Hữu Phúc,689,male,45,Nội trợ/Sinh viên,1927416,5000000,Phường Bến Thành (Q1),Đồng Nai,3,1,3,2,True,14/05/2025,3255569,29/07/2021,False,Mass,63,Gold,mobile,0.2664,Low,2
2,3,Lê Văn Mai,702,female,44,Chủ Doanh nghiệp nhỏ,304931745,109000000,Phường Hòa Bình (Q11),TP. Hồ Chí Minh,4,0,2,8,False,10/09/2021,0,03/03/2021,False,Priority,36,Silver,offline,0.1343,Low,4
3,4,Dương Trần Nhiên,766,male,44,Chủ Doanh nghiệp nhỏ,50615501,79000000,Phường 7 (Q7),TP. Hồ Chí Minh,3,0,4,3,False,04/03/2025,0,12/02/2022,False,Priority,23,Bronze,offline,0.2185,Low,4
4,5,Dương Thu Linh,677,female,77,Giáo viên/Giảng viên,40532432,25000000,Phường Thắng Lợi (Q10),TP. Hồ Chí Minh,2,1,4,3,False,06/08/2022,0,24/07/2022,False,Emerging,23,Bronze,offline,0.2942,Low,2
5,6,Phạm Minh Lan,708,female,60,Kỹ sư/Chuyên viên IT,36274615,57000000,Thị trấn Tân Túc (BC),Bình Dương,3,1,1,6,False,29/01/2023,0,15/11/2021,False,Affluent,26,Bronze,offline,0.1850,Low,2
6,7,Mai Ngọc Thành,677,male,41,Nội trợ/Sinh viên,24152499,14000000,Xã Xuân Thới Đông (HM),TP. Hồ Chí Minh,3,1,2,4,False,23/07/2023,0,18/07/2021,True,Mass,21,Bronze,offline,0.3099,Low,2
7,8,Hoàng Kim Dương,690,female,56,Hưu trí,20137060,19000000,Phường 14 (Q12),Bình Dương,4,1,2,4,True,02/05/2025,6462745,30/03/2021,False,Emerging,64,Gold,mobile,0.1841,Low,2
8,9,Lê Gia Khoa,728,male,51,Nội trợ/Sinh viên,12678179,9000000,Xã Vĩnh Lộc A (BC),Tỉnh khác,2,1,1,5,False,06/05/2023,0,15/02/2023,True,Mass,22,Bronze,offline,0.2922,Low,2
9,10,Nguyễn Thu Linh,711,female,26,Chủ Doanh nghiệp nhỏ,302651881,96000000,Phường Tân Định (Q1),TP. Hồ Chí Minh,3,1,3,8,False,20/06/2022,0,29/01/2022,False,Priority,39,Silver,offline,0.1344,Low,4


In [6]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

df.to_sql("bank_churn", engine, if_exists="replace", index=False)
print(f"Загружено {len(df)} строк в PostgreSQL")

Загружено 80000 строк в PostgreSQL


In [7]:
query = """
SELECT
    customer_segment,
    COUNT(*) AS total,
    ROUND(AVG(CAST(exit AS INT)) * 100, 2) AS churn_rate_pct
FROM bank_churn
GROUP BY customer_segment
ORDER BY churn_rate_pct DESC
"""

print("Churn rate по сегментам (SQL-запрос):")
print(pd.read_sql(query, engine))

Churn rate по сегментам (SQL-запрос):
  customer_segment  total  churn_rate_pct
0             Mass  21436           39.48
1         Emerging  31662           17.62
2         Affluent  11210            3.10
3         Priority  15692            0.07


## Пропуски и дубликаты

Данные уже подготовлены — пропусков нет ни в одной колонке. Но проверю на дубликаты: иногда подготовка оставляет задвоенные строки.

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"пропусков": missing, "%": missing_pct})
missing_summary = missing_summary[missing_summary["пропусков"] > 0].sort_values("%", ascending=False)

print("Колонки с пропусками:")
print(missing_summary if len(missing_summary) else "Пропусков нет")

print("\nПолных дубликатов строк:", df.duplicated().sum())
print("Дубликатов по id:", df["id"].duplicated().sum())

## Описательная статистика

Смотрю на минимумы и максимумы — там обычно всплывают аномалии: отрицательный баланс, нереалистичный возраст, кредитный скор вне допустимого диапазона.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].describe().round(2).T

## Проверка диапазонов

Кредитный скор обычно в диапазоне 300–850, возраст — реалистичные границы, баланс не должен быть отрицательным (если это не овердрафт).

In [ ]:
checks = {
    "credit_sco вне 300-850": ((df["credit_sco"] < 300) | (df["credit_sco"] > 850)).sum(),
    "age < 18 или > 100": ((df["age"] < 18) | (df["age"] > 100)).sum(),
    "balance < 0": (df["balance"] < 0).sum(),
    "tenure_ye < 0": (df["tenure_ye"] < 0).sum(),
    "nums_card < 0": (df["nums_card"] < 0).sum(),
    "engagement_score вне 0-100": ((df["engagement_score"] < 0) | (df["engagement_score"] > 100)).sum(),
}

for check, count in checks.items():
    status = "есть" if count > 0 else "ок"
    print(f"[{status:>4}]  {check}: {count}")

## Выбросы по IQR

Выброс — не всегда ошибка. Но надо понимать, сколько их и в каких колонках, прежде чем строить модель: некоторые модели чувствительны к выбросам, деревья — устойчивы.

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

outlier_summary = pd.Series({col: iqr_outliers(df[col]) for col in numeric_cols})
outlier_summary = outlier_summary[outlier_summary > 0].sort_values(ascending=False)
print("Колонки с выбросами (IQR-метод):")
print(outlier_summary)

## Согласованность полей

Проверяю логические связи: например, `active_member = False`, но при этом есть недавняя активность — противоречие. Или дата активности раньше даты создания аккаунта — такого быть не может.

In [ ]:
df["last_active_date"] = pd.to_datetime(df["last_active_date"], errors="coerce")
df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")

print("Пропуски после конвертации:")
print(f"  last_active_date: {df['last_active_date'].isna().sum()}")
print(f"  created_date:     {df['created_date'].isna().sum()}")

recent_but_inactive = df[
    (df["active_member"] == False) &
    (df["last_active_date"] > df["last_active_date"].max() - pd.Timedelta(days=30))
]
print(f"\nactive_member=False, но активность за последние 30 дней: {len(recent_but_inactive)}")

negative_tenure = df[df["last_active_date"] < df["created_date"]]
print(f"last_active_date раньше created_date: {len(negative_tenure)}")

## Распределение целевой переменной

Целевая переменная — `exit` (ушёл клиент или нет). Смотрю общий churn rate и разбивку по сегментам: если классы сильно несбалансированы, при моделировании понадобится `class_weight='balanced'`.

In [ ]:
churn_rate = df["exit"].mean()
print(f"Общий churn rate: {churn_rate:.2%}")
print(f"\nРаспределение по сегментам:")
print(df.groupby("customer_segment")["exit"].agg(["count", "mean"]).rename(columns={"mean": "churn_rate"}).round(4))

## Итоги аудита

- **Размер:** 80 000 клиентов, 26 признаков.
- **Пропуски:** нет ни в одной колонке, кроме `last_active_date`.
- **`last_active_date`:** 49 471 строка (62%) без значения. Данные есть только у активных клиентов, поэтому использовать колонку в анализе рискованно — выборка смещена. В модель оттока не пойдёт.
- **Дубликаты:** полных дубликатов строк нет, дубликатов по `id` нет.
- **Аномалии по диапазонам:** проверки `credit_sco` (300–850), `age` (18–100), `balance`, `tenure_ye`, `nums_card`, `engagement_score` прошли без срабатываний.
- **Выбросы по IQR:** больше всего в `married`, `last_transaction_month`, `cluster_group`, `engagement_score`, `monthly_ir` — это не ошибки, а реальные клиенты с нетипичным поведением. Для деревьев решений не критично.
- **Churn rate:** 18% по всей выборке, но распределение по сегментам крайне неравномерное:

| Сегмент | Клиентов | Churn rate |
|---|---|---|
| Mass | 21 436 | **39.5%** |
| Emerging | 31 662 | 17.6% |
| Affluent | 11 210 | 3.1% |
| Priority | 15 692 | **0.07%** |

Mass — 27% клиентской базы, но даёт больше половины всех уходов. Разница с Priority — в 560 раз. Это ключевая гипотеза для следующих ноутбуков: что именно в Mass-сегменте приводит к оттоку.

Данные готовы к анализу. Единственная проблемная колонка — `last_active_date`, но она не входит в модель.